Підготовка до виконання роботи

In [1]:
import sys
!{sys.executable} -m pip install pandas requests


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Імпорт необхідних бібліотек для роботи з даними та мережевими запитами

In [2]:
import pandas as pd
import requests
import os
from datetime import datetime

Завантаження даних.
Реалізація процедури завантаження файлів VHI з сервера NOAA та збереження їх у локальну папку

In [3]:
def download_vhi_data():
    if not os.path.exists('vhi_data'):
        os.makedirs('vhi_data')

    url_template = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={}&year1=1981&year2=2024&type=Mean"

    for i in range(1, 28):
        # Назва файлу містить ID області та мітку часу
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        filename = f"vhi_data/vhi_id_{i}_{timestamp}.csv"

        # Перевірка на дублікати (якщо файл для цієї області вже є, не качаємо знову)
        if any(f.startswith(f"vhi_id_{i}_") for f in os.listdir('vhi_data')):
            print(f"Файл для області {i} вже існує. Пропускаю.")
            continue

        print(f"Завантажую область {i}...")
        response = requests.get(url_template.format(i))

        # Очищення від HTML-тегів, які іноді приходять з сервера
        content = response.text.replace('<pre>', '').replace('</pre>', '')

        with open(filename, 'w') as f:
            f.write(content)

download_vhi_data()

Файл для області 1 вже існує. Пропускаю.
Файл для області 2 вже існує. Пропускаю.
Файл для області 3 вже існує. Пропускаю.
Файл для області 4 вже існує. Пропускаю.
Файл для області 5 вже існує. Пропускаю.
Файл для області 6 вже існує. Пропускаю.
Файл для області 7 вже існує. Пропускаю.
Файл для області 8 вже існує. Пропускаю.
Файл для області 9 вже існує. Пропускаю.
Файл для області 10 вже існує. Пропускаю.
Файл для області 11 вже існує. Пропускаю.
Файл для області 12 вже існує. Пропускаю.
Файл для області 13 вже існує. Пропускаю.
Файл для області 14 вже існує. Пропускаю.
Файл для області 15 вже існує. Пропускаю.
Файл для області 16 вже існує. Пропускаю.
Файл для області 17 вже існує. Пропускаю.
Файл для області 18 вже існує. Пропускаю.
Файл для області 19 вже існує. Пропускаю.
Файл для області 20 вже існує. Пропускаю.
Файл для області 21 вже існує. Пропускаю.
Файл для області 22 вже існує. Пропускаю.
Файл для області 23 вже існує. Пропускаю.
Файл для області 24 вже існує. Пропускаю.
Ф

Обробка та очищення даних.
Читання завантажених файлів, формування єдиного DataFrame, додавання назв областей та очищення значень року

In [4]:
def create_clean_df(folder):
    # Словник для заміни індексів на український алфавіт
    # (Це приклад, переконайся, що порядок відповідає вимогам викладача)
    # NOAA ID : Назва (і новий індекс буде за порядком у списку)
    area_names = {
        1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська",
        6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська", 9: "Київська", 10: "Кіровоградська",
        11: "Луганська", 12: "Львівська", 13: "Миколаївська", 14: "Одеська", 15: "Полтавська",
        16: "Рівненська", 17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
        21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська", 25: "Республіка Крим"
    }

    files = [f for f in os.listdir(folder) if f.endswith('.csv')]
    all_frames = []

    for f in files:
        # Читаємо файл, ігноруючи заголовки NOAA
        temp_df = pd.read_csv(f"{folder}/{f}", index_col=False, header=1,
                              names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'])

        # Видаляємо сміття (рядки з текстом)
        temp_df = temp_df.dropna()

        # Дістаємо старий ID з назви файлу
        old_id = int(f.split('_')[2])

        # Додаємо нові стовпці
        temp_df['area_index'] = old_id # Тут можна реалізувати логіку переіндексації
        temp_df['area_name'] = area_names.get(old_id, "Невідомо")

        all_frames.append(temp_df)

    return pd.concat(all_frames).reset_index(drop=True)

df = create_clean_df('vhi_data')
df['Year'] = df['Year'].astype(str).str.replace('<tt>', '', regex=False).str.replace('</tt>', '', regex=False)
df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,area_index,area_name
0,1982,1.0,0.059,258.24,51.11,48.78,49.95,10,Кіровоградська
1,1982,2.0,0.063,261.53,55.89,38.20,47.04,10,Кіровоградська
2,1982,3.0,0.063,263.45,57.30,32.69,44.99,10,Кіровоградська
3,1982,4.0,0.061,265.10,53.96,28.62,41.29,10,Кіровоградська
4,1982,5.0,0.058,266.42,46.87,28.57,37.72,10,Кіровоградська


Процедури для формування вибірок

In [5]:
# 1. Ряд VHI для області за вказаний рік
def get_vhi_by_year(df, area_id, year):
    return df[(df['area_index'] == area_id) & (df['Year'].astype(str) == str(year))]

# 2. Ряд VHI за діапазон років для вказаних областей
def get_vhi_range(df, areas, start, end):
    return df[(df['area_index'].isin(areas)) & (df['Year'].astype(int).between(int(start), int(end)))]

# 3. Екстремуми
def get_min_max(df, area_id, year):
    data = get_vhi_by_year(df, area_id, year)
    return {"max": data['VHI'].max(), "min": data['VHI'].min()}

In [6]:
print("Ряд VHI для області 10 за 2000 рік:")
vhi_2000 = get_vhi_by_year(df, 10, 2000)
print(vhi_2000.head())

Ряд VHI для області 10 за 2000 рік:
     Year  Week    SMN     SMT    VCI    TCI    VHI  area_index  \
936  2000   1.0  0.033  260.29  21.28  40.64  30.96          10   
937  2000   2.0  0.033  260.40  22.74  42.06  32.40          10   
938  2000   3.0  0.036  261.40  28.74  39.77  34.26          10   
939  2000   4.0  0.043  262.45  36.16  37.55  36.86          10   
940  2000   5.0  0.049  264.25  39.26  35.74  37.50          10   

          area_name  
936  Кіровоградська  
937  Кіровоградська  
938  Кіровоградська  
939  Кіровоградська  
940  Кіровоградська  


In [7]:
print("Максимум та мінімум VHI для області 10 за 2000 рік:")
extremes = get_min_max(df, 10, 2000)
print(extremes)

Максимум та мінімум VHI для області 10 за 2000 рік:
{'max': np.float64(65.0), 'min': np.float64(20.97)}


In [8]:
print("Дані для областей 10 та 12 за 1990-1995 роки:")
range_data = get_vhi_range(df, [10, 12], 1990, 1995)
print(range_data.head())

Дані для областей 10 та 12 за 1990-1995 роки:
     Year  Week    SMN     SMT    VCI   TCI    VHI  area_index       area_name
416  1990   1.0  0.080  270.32  73.82  3.56  38.69          10  Кіровоградська
417  1990   2.0  0.085  271.50  78.61  1.46  40.03          10  Кіровоградська
418  1990   3.0  0.089  272.22  82.76  1.01  41.88          10  Кіровоградська
419  1990   4.0  0.096  273.42  87.60  0.25  43.92          10  Кіровоградська
420  1990   5.0  0.106  275.31  88.48  0.04  44.26          10  Кіровоградська


Аналіз енергоспоживання домогосподарств (Частина 2).
Завантаження даних, очищення від пропусків та приведення типів

In [9]:
df_power = pd.read_csv('household_power_consumption.txt', sep=';', low_memory=False, na_values='?')
df_power = df_power.dropna()

float_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage',
              'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df_power[float_cols] = df_power[float_cols].astype(float)

print("Дані для Частини 2 завантажено!")
df_power.head()


Дані для Частини 2 завантажено!


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


Виконання аналітичних запитів до датасету (Завдання 1-3).
Потужність > 5 кВт, Струм 19-20 А, Випадкова вибірка 500 000 записів та розрахунок середніх значень

In [10]:
import timeit

# 1. Потужність > 5 кВт
def task_1():
    return df_power[df_power['Global_active_power'] > 5]

# 2. Струм 19-20 А + пралка/холодильник > бойлер/кондиціонер
def task_2():
    return df_power[(df_power['Global_intensity'].between(19, 20)) &
                    ((df_power['Sub_metering_1'] + df_power['Sub_metering_2']) > df_power['Sub_metering_3'])]

# 3. Випадкові 500 000 записів та середнє
def task_3():
    sample = df_power.sample(n=500000, replace=False)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

# Вимірюємо час та виводимо результати
print("Час виконання завдання 1:", timeit.timeit(task_1, number=1), "сек")
print("Знайдено рядків:", len(task_1()))

print("\nЧас виконання завдання 2:", timeit.timeit(task_2, number=1), "сек")
print("Знайдено рядків:", len(task_2()))

print("\nЧас виконання завдання 3:", timeit.timeit(task_3, number=1), "сек")
print("Середні значення:\n", task_3())

Час виконання завдання 1: 0.00394610000148532 сек
Знайдено рядків: 17547

Час виконання завдання 2: 0.016100499997264706 сек
Знайдено рядків: 5579

Час виконання завдання 3: 0.1257735999970464 сек
Середні значення:
 Sub_metering_1    1.132260
Sub_metering_2    1.318380
Sub_metering_3    6.460468
dtype: float64


In [11]:
import sys
!{sys.executable} -m pip install scipy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Складна фільтрація, статистичний аналіз та трансформація даних (Завдання 4-7).
Вечірня вибірка з проріджуванням, нормалізація, кореляційний аналіз та One Hot Encoding

In [12]:
import datetime

# 4. Складна вибірка за часом та проріджування результатів
def task_4():
    df_temp = df_power.copy()
    # Перетворюємо колонку Time, щоб Python розумів години/хвилини
    df_temp['Time'] = pd.to_datetime(df_temp['Time'], format='%H:%M:%S').dt.time
    limit_time = datetime.time(18, 0, 0)

    # Фільтруємо: вечір + велика потужність
    res = df_temp[(df_temp['Time'] > limit_time) & (df_temp['Global_active_power'] > 6)]

    # Умова: група 2 (праля) споживає найбільше серед усіх трьох груп
    res = res[(res['Sub_metering_2'] > res['Sub_metering_1']) & (res['Sub_metering_2'] > res['Sub_metering_3'])]

    # Беремо кожен 3-й рядок з першої половини та кожен 4-й з другої
    half = len(res) // 2
    first_part = res.iloc[:half:3]
    second_part = res.iloc[half::4]
    return pd.concat([first_part, second_part])

# 5. Нормалізація та стандартизація
def task_5():
    target = df_power[['Global_active_power', 'Global_intensity']].copy()
    # Min-Max нормалізація (в межах від 0 до 1)
    norm = (target - target.min()) / (target.max() - target.min())
    # Стандартизація (середнє = 0, відхилення = 1)
    std = (target - target.mean()) / target.std()
    return norm, std

# 6. Коефіцієнти кореляції (Пірсон та Спірмен)
def task_6():
    p = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='pearson')
    s = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='spearman')
    return p, s

# 7. One Hot Encoding для дати (беремо шматочок для прикладу)
def task_7():
    return pd.get_dummies(df_power['Date'].head(100))

# Виклик функцій для перевірки
print("--- Результати останніх завдань ---")
print(f"Завдання 4: Знайдено {len(task_4())} рядків після відбору.")
p_corr, s_corr = task_6()
print(f"Завдання 6: Кореляція Пірсона = {p_corr:.4f}, Спірмена = {s_corr:.4f}")
print("\nЗавдання 7 (фрагмент One Hot Encoding):")
print(task_7().head())

--- Результати останніх завдань ---
Завдання 4: Знайдено 310 рядків після відбору.
Завдання 6: Кореляція Пірсона = 0.9989, Спірмена = 0.9954

Завдання 7 (фрагмент One Hot Encoding):
   16/12/2006
0        True
1        True
2        True
3        True
4        True
